# 建置爬蟲核心工具箱 (Environment Setup)

本單元引入爬蟲三大支柱：**Requests** (模擬請求)、**BeautifulSoup** (解析 HTML) 、 **JSON** (資料交換)與**re** (正則表達式)。<br>
我們將使用 `requests` 與 PTT 伺服器通訊，並指定高效的 `lxml` 引擎協助 `BeautifulSoup` 提取標題與內文。<br>
請跟著老師填寫下方程式碼，完成自動化爬蟲的環境初始化。

In [1]:
# 若尚未安裝需要的套件，可在終端或 notebook 執行：
# !pip install requests beautifulsoup4 lxml

import requests         # 發送 HTTP 請求，用來從網站抓取網頁內容
from bs4 import BeautifulSoup as bs  # 解析 HTML / XML，將字串轉成可以操作的樹狀結構
import lxml             # 上課用的解析器（解析器需已安裝）
import time             # 取得目前時間，或讓程式暫停一段時間，做延遲，避免被 PTT 封鎖
import json             # 處理 JSON 格式的資料（例如 API 回傳或儲存設定）
import re               # 正規表達式

在開始爬蟲之前，我們需要先模擬瀏覽器的行為。首先，我們將 PTT 股票版的搜尋網址（包含關鍵字「台達電」）準備好，接著使用 requests.get() 向伺服器敲門並索取資料。確認伺服器回應成功（狀態碼 200）後，我們再利用 BeautifulSoup 這個強大的工具，將雜亂的 HTML 原始碼解析成容易提取的物件。

In [3]:
# 構造 GET 請求網址：`?` 宣告參數開始，`&` 用來連接多個條件
url = 'https://www.ptt.cc/bbs/Stock/search?page=1&q=台達電'

# 發送 GET 請求到該網址
# 提示：requests 會回傳一個 Response 物件。
res = requests.get(url)

# 簡單檢查是否成功收到回應（HTTP 200 表示成功）
if res.status_code != 200:
    raise RuntimeError(f'取得網頁失敗，HTTP 狀態碼：{res.status_code}')

# 使用 BeautifulSoup 解析回傳的 HTML（使用 lxml 解析器，需要先安裝 lxml）
# res.text 就是你在瀏覽器按右鍵「檢視網頁原始碼」看到的那些 HTML 內容
soup = bs(res.text, 'lxml')


接下來是爬蟲最關鍵的一步：資料定位。請大家在瀏覽器按 F12 觀察，會發現 PTT 列表中的每一篇文章，其實都是一個 class 為 r-ent 的 div 區塊。我們使用 BeautifulSoup 的 .select() 方法，像篩子一樣把所有符合 div.r-ent 特徵的區塊都撈出來，存成一個列表 (List)。

In [4]:
# 使用 CSS 選擇器選取所有文章區塊
# PTT 文章列表中每篇文章的外層容器，元素組成: <div class="r-ent"> ... </div>
post_entries = soup.select('div.r-ent')

# 檢查點 : 印出數量看有沒有抓到東西
print(f"找到{len(post_entries)}篇文章")

找到20篇文章


> **教學註記**：為優化課堂節奏，此函式已預先封裝供直接調用。
> **核心機制**：
> 1. **實例化**：執行 `datetime.now()` 捕捉系統當下時間點。
> 2. **屬性存取**：透過 `.month` 動態提取月份整數，實現自動化更新。
> 3. **month屬性**：呼叫month_extract()函式，得系統偵測的月份(整數，Ex:1 | 12)

In [5]:
from datetime import datetime

'''
練習如何從datetime物件動態提取月份，並傳遞給month使用
'''
def month_extract():
    # 1. 獲取當前時間物件 (Instantiation)
    # 這是動態的源頭，每次執行時都會抓取系統當下時間
    current_time: datetime = datetime.now()

    # 2. 提取屬性 (Attribute Extraction)
    # 找month屬性 => .month
    # 注意 : 屬性存取，不需要括號 ()
    This_month : int = current_time.month

    # 3.  驗證輸出
    # print(f"系統偵測時間:{current_time}")
    # print(f"提取月份:{This_month}")
    return This_month
if __name__ == "__main__":
    month_extract()
    

### 任務：實作「上月文章過濾器」

我們已經抓到了所有文章區塊 (`post_entries`)，現在需要一個過濾器，只留下**上個月份**的資料。

**實作要求：**
1.  **邏輯健壯性**：程式必須能正確處理 **1月份執行時** 的狀況（需自動回推至去年 12月），不能報錯或抓不到資料。
2.  **精準匹配**：請使用 `startswith` 搭配 `/` 符號，確保不會將 1 月誤判為 11 月。
3.  **輸出結果**：遍歷列表，印出所有符合條件的日期字串（需去除多餘空白）。

*請開始 Coding！*


In [31]:
# month_extract() 已經取得當前月份
# 為甚麼需要寫這個if判斷? 請學員討論
if int(month_extract()) == 1:
    last_month = 12
else:
    last_month = month_extract() -1

#Pattern(匹配的格式)，我們預期要匹配到"1/"、"2/"、...、"12/"
# f-string（格式化字串）：在字串前加 f，使用 {變數} 直接嵌入變數值，簡潔替代 str.format()
pattern = f"{last_month}/"

#.strip():去除前後空白
for post in post_entries:
    date_tag = post.find(class_="date")
    date_text = date_tag.text 
    if date_text.strip().startswith(pattern):
        print(date_text.strip())

11/14
11/13
11/13
11/10
11/10
11/06


### 任務：打造網址收集器 (URL Collector)

我們已經成功篩選出上個月的文章了，現在下一步是**把這些文章的連結存起來**，這樣我們等一下才能進去每一篇文章爬內文。

**請依照以下邏輯完成程式碼：**

#### 1. 準備工作
*   建立一個空的列表 (List)，取名為 `target_urls`，用來存放最終結果。
*   建立一個變數 `base_url`，內容為 `"https://www.ptt.cc"`。
    *   *思考：為什麼需要這個變數？直接用爬下來的連結不行嗎？*

#### 2. 擷取連結 (在迴圈內執行)
當日期符合條件時，我們要進行以下動作：
*   **定位標籤**：找到文章區塊內的 `<a>` 標籤（代表超連結）。
*   **取得屬性**：使用 `post.a['href']` 語法，抓取該標籤的 `href` 屬性。這通常會是一個**相對路徑** (例如 `/bbs/Stock/...`)。

#### 3. 補全與儲存
*   **拼接網址**：將 `base_url` 與剛剛抓到的相對路徑相加，變成可以點擊的完整網址。
*   **加入列表**：使用 `.append()` 方法，將完整網址存入 `target_urls`。

#### 4. 驗收
*   迴圈結束後，印出 `target_urls`，確認裡面是不是都是完整的 `https://...` 網址。


In [7]:
# 欲取得上個月的每一篇討論文章，我們首先要取得的就是其個別的網址
# 1. 初始化容器：準備一個空列表來裝我們抓到的網址
target_urls = [] 

# PTT主網域變數
# 2. 定義主網域：PTT 的連結通常是「相對路徑」，需要補上開頭

base_url = 'https://www.ptt.cc'

'''
<a href="/bbs/Stock/M.1764646716.A.193.html">[新聞] 台達電營運 大摩按讚</a>
這是我們的目標元素
'''
# post.a 代表找到該區塊內的第一個<a>超連結標籤
# ['href']是像查字典一樣，取出該標籤的href屬性值
# 字串拼接：主網域 + 相對路徑 = 完整網址
# 將結果加入列表
for post in post_entries:
    date_tag = post.find(class_="date")
    date_text = date_tag.text

    if date_text.strip().startswith(pattern):
       post_url = post.a['href']
       print(post_url)
       complete_url = base_url + post_url
       target_urls.append(complete_url)
print(target_urls)


/bbs/Stock/M.1763098884.A.028.html
/bbs/Stock/M.1763019039.A.77F.html
/bbs/Stock/M.1763013474.A.A00.html
/bbs/Stock/M.1762761036.A.D8F.html
/bbs/Stock/M.1762759720.A.2A5.html
/bbs/Stock/M.1762392173.A.10F.html
['https://www.ptt.cc/bbs/Stock/M.1763098884.A.028.html', 'https://www.ptt.cc/bbs/Stock/M.1763019039.A.77F.html', 'https://www.ptt.cc/bbs/Stock/M.1763013474.A.A00.html', 'https://www.ptt.cc/bbs/Stock/M.1762761036.A.D8F.html', 'https://www.ptt.cc/bbs/Stock/M.1762759720.A.2A5.html', 'https://www.ptt.cc/bbs/Stock/M.1762392173.A.10F.html']


### 任務：儲存多篇文章資料

我們現在要進入每一篇文章抓取詳細資料。請大家想像：
*   **每一篇文章** 就像一張「個人資料卡」(Dictionary)。
*   我們需要一本「通訊錄」(List) 來把這些卡片全部裝起來。

**請依照以下結構撰寫程式：**

1.  **建立總容器**：
    *   在迴圈外面，建立一個空列表 `articles_data = []`。

2.  **單篇文章處理 (在迴圈內)**：
    *   發送請求 (`requests`) 並解析 (`BeautifulSoup`)。
    *   抓取標題 (`title`)。
    *   **關鍵步驟**：建立一個字典 `article_info`，把標題和網址存進去。
        *   範例：`article_info = {"title": title, "url": target_url}`

3.  **歸檔**：
    *   使用 `.append()` 把這篇文章的字典，放入總容器 `articles_data` 中。

In [ ]:
from time import sleep
articles_list = []
# 建立「單篇文章」的字典(Dict)
# 把「單篇文章」的字典裝進大列表裡面
for target_url in target_urls:
    try:
        res = requests.get(target_url)
        soup = bs(res.text, 'lxml')
        meta_value = soup.select('span.article-meta-value')
        if len(meta_value) >=3:
            title = meta_value[2].text
        else:
            title = "無標題或格式錯誤"
        
        if len(meta_value) >=4:
            time = meta_value[3].text
        else:
            raise Exception("無時間或格式錯誤")
        
        comment = soup.select('span.f3.push-content')
        comment_list = []
        for c in comment:
            clean_data= c.text
            comment_list.append(clean_data)

        # PTT (批踢踢實業坊)，它的HTML結構並非現代化的語意標籤 (Semantic HTML)，而是由舊式BBS系統轉換而來
        # 這導致「內文」並不是包在一個乾淨的 <p> 或 <article-body> 標籤中
        # 而是作為「文字節點 (Text Node)」散落在 #main-content元素內，與標題、時間、推文混雜在一起
        # 這裡我們不需要保留「標題列」或「時間」，我們使用beautifulsoup的decomposed()方法將不需要的元素去除
        
        # 步驟一 : 鎖定主內容元素
        main_content = soup.select_one('#main-content')
        
        if main_content:
            # 步驟二: 移除不要的元素
            # 批次處理：利用 CSS Selector 的逗號 , 功能，一次選取所有不要的類別（Meta + Push），一個迴圈解決，不需要寫好幾個迴圈。
            # 不要的元素(1) : .article-metaline (標題、作者、時間)
            # 不要的元素(2) : .article-metaline-right (看板名稱)
            # 不要的元素(3) : span (零散資訊)
            # 不要的元素(4) : div.push (推文區塊)
            unwanted_elements = main_content.select('.article-metaline, .article-metaline-right, span, div.push')
            
            #步驟三: 執行銷毀
            for element in unwanted_elements:
                element.decompose()

        content = main_content.text.strip()
        clean_content = content.replace('\n', ' ')

        article_info={
            "title": title,
            "url": target_url,
            "time": time,
            "content": clean_content,
            "comments": comment_list,

        }
        articles_list.append(article_info)
        print(f"已成功爬取{title}")
        sleep(1)
    except Exception as e:
        print(f"爬取失敗:{target_url}，錯誤原因:{e}")
        

print(articles_list) 


已成功爬取Re: [標的] 2308台達電
已成功爬取Re: [請益] 2308台達電
已成功爬取[請益] 2308台達電
已成功爬取[新聞] 台達電10月營收573.84億元創新高 前10月
已成功爬取[情報] 2308 台達電 10月營收
已成功爬取[新聞] 台達電取得全球 ICT 產業首張 ISO 59004
[{'title': 'Re: [標的] 2308台達電', 'url': 'https://www.ptt.cc/bbs/Stock/M.1763098884.A.028.html', 'time': 'Fri Nov 14 13:41:21 2025', 'content': '自己的標的自己回 不囉嗦先上圖  https://i.imgur.com/yoxEQQZ.jpeg 大概從9月初抱到今天 沒有吃到關稅戰時的紅利 買的價格也不算低 當時給自己下的目標是靠近1100時出場 但 就是一個貪字 以為可以穩穩破 結果 就是今天開盤趕快出場 大概少賺30萬 這幾天尾盤都有看到有人拋 未來如果修正到800元左右 考慮會再進場，近期就等抽鴻勁吧 ---- Sent from  on my iPhone 16 Pro  --', 'comments': [': 恭喜 還是賺爛了', ': 先跑', ': 台達電已經炒到大後年了吧，獲利出場可以', ': 水啦 這筆錢拿去抽鴻勁 抽到再賺70萬', ': 貪心就是一個警訊 我看上上下下覺得不適合我 我就', ': 出場了10/22賣光台美股 結果費半狂噴又進場 套牢獲', ': 利噴光qq', ': 這檔要殺 都殺超深的', ': 漲過頭會修正很久', ': https://i.imgur.com/BmqcfCn.jpeg', ': 留意', ': 散戶心態:既要且要', ': 華城第二', ': 魚尾給我賺，感謝']}, {'title': 'Re: [請益] 2308台達電', 'url': 'https://www.ptt.cc/bbs/Stock/M.1763019039.A.77F.html', 'time': 'Thu Nov 13 15:30:37 2025', 'content': '跟廠務一樣，創高後一路崩 就只是資金不在那而已 看看美股ai基礎建設營收都創高，每個都崩20-30% 台達電看來

In [30]:
Output = json.dumps(articles_list, ensure_ascii=False, indent=4)
print(Output)
with open('ptt_stock_articles.json', 'w', encoding='utf-8') as f:
    f.write(Output)

[
    {
        "title": "Re: [標的] 2308台達電",
        "url": "https://www.ptt.cc/bbs/Stock/M.1763098884.A.028.html",
        "time": "Fri Nov 14 13:41:21 2025",
        "content": "自己的標的自己回 不囉嗦先上圖  https://i.imgur.com/yoxEQQZ.jpeg 大概從9月初抱到今天 沒有吃到關稅戰時的紅利 買的價格也不算低 當時給自己下的目標是靠近1100時出場 但 就是一個貪字 以為可以穩穩破 結果 就是今天開盤趕快出場 大概少賺30萬 這幾天尾盤都有看到有人拋 未來如果修正到800元左右 考慮會再進場，近期就等抽鴻勁吧 ---- Sent from  on my iPhone 16 Pro  --",
        "comments": [
            ": 恭喜 還是賺爛了",
            ": 先跑",
            ": 台達電已經炒到大後年了吧，獲利出場可以",
            ": 水啦 這筆錢拿去抽鴻勁 抽到再賺70萬",
            ": 貪心就是一個警訊 我看上上下下覺得不適合我 我就",
            ": 出場了10/22賣光台美股 結果費半狂噴又進場 套牢獲",
            ": 利噴光qq",
            ": 這檔要殺 都殺超深的",
            ": 漲過頭會修正很久",
            ": https://i.imgur.com/BmqcfCn.jpeg",
            ": 留意",
            ": 散戶心態:既要且要",
            ": 華城第二",
            ": 魚尾給我賺，感謝"
        ]
    },
    {
        "title": "Re: [請益] 2308台達電",
        "url": "https://www.ptt.cc/bbs/Stock/M.1763019039.A.77F.html",
        "time